# Import Required Libraries

In [0]:
from pyspark.sql import functions as F

# Defining Start and end dates

In [0]:
# start date and end date
start_date = "2024-01-01"
end_date   = "2025-12-01"

In [0]:
# 1️⃣ Generate one row per month start between start_date and end_date

df = (
    spark.sql(
        f"""
        select explode(
            sequence(
                to_date('{start_date}'),
                to_date('{end_date}'),
                interval 1 month
            )
        ) as month_start_date
    """)
)

In [0]:
df.limit(5).display()

# Add useful analytics columns

In [0]:
df = (
  df.withColumn("date_key",F.date_format("month_start_date","yyyyMM").cast("int"))\
  .withColumn("year",F.year("month_start_date"))\
  .withColumn("Month_name",F.date_format("month_start_date","MMMM"))\
  .withColumn("Month_sort_name",F.date_format("month_start_date","MMM"))\
  .withColumn("Quarter",F.concat(F.lit("Q"),F.quarter("month_start_date")))\
  .withColumn("Year_quarter",F.concat(F.col("year"),F.lit("-"),F.col("Quarter")))
)


In [0]:
df.limit(5).display()

In [0]:
# Save as a table
df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("FMCG.gold.dim_date")